In [ ]:
 # ==============================================================================
# SECTION 1: GOOGLE DRIVE + SINTELA H5 DISCOVERY (COLAB-ONLY)
# Purpose:
#   1) Mount Drive safely
#   2) Find your .h5 file anywhere under MyDrive
#   3) Inspect H5 structure
#   4) Auto-detect a likely DAS dataset (2D, large)
# ==============================================================================

import os
import h5py
import numpy as np
from google.colab import drive

# -----------------------------
# 1) Mount Drive (safe)
# -----------------------------
MOUNT_POINT = "/content/drive"

if not os.path.exists(MOUNT_POINT):
    os.makedirs(MOUNT_POINT, exist_ok=True)

# If already mounted, don't remount (prevents "mountpoint contains files")
if not os.path.ismount(MOUNT_POINT):
    drive.mount(MOUNT_POINT)
else:
    print(f"Drive already mounted at {MOUNT_POINT}")

# -----------------------------
# 2) Locate H5 file anywhere in MyDrive
# -----------------------------
SEARCH_ROOT = "/content/drive/MyDrive/FiberMind/DAS_Data"
TARGET_FILENAME = "10_44_45.h5"   # change if needed

def find_file(root, filename):
    for dirpath, dirnames, filenames in os.walk(root):
        if filename in filenames:
            return os.path.join(dirpath, filename)
    return None

FILE_PATH = find_file(SEARCH_ROOT, TARGET_FILENAME)

if FILE_PATH is None:
    raise FileNotFoundError(
        f"Could not find {TARGET_FILENAME} under {SEARCH_ROOT}.\n"
        f"Tip: confirm it's in Drive and the exact filename matches (case-sensitive)."
    )

print("✅ Found H5 file:", FILE_PATH)

# -----------------------------
# 3) Print H5 tree (compact)
# -----------------------------
def print_h5_tree(f, max_items_per_level=50):
    def _recurse(name, obj, indent=0):
        pad = "  " * indent
        if isinstance(obj, h5py.Group):
            print(f"{pad}📁 {name}/")
        else:
            shape = obj.shape
            dtype = obj.dtype
            chunks = obj.chunks
            print(f"{pad}🧊 {name}  shape={shape}  dtype={dtype}  chunks={chunks}")

    count = 0
    def visitor(name, obj):
        nonlocal count
        if count >= 1000:
            return
        _recurse(name, obj, indent=name.count("/"))
        count += 1

    f.visititems(visitor)

# -----------------------------
# 4) Auto-detect likely DAS dataset
#    Criteria: 2D dataset, large, numeric
# -----------------------------
def pick_likely_das_dataset(f):
    candidates = []
    def visitor(name, obj):
        if isinstance(obj, h5py.Dataset):
            if obj.ndim == 2 and np.issubdtype(obj.dtype, np.number):
                # Score by size (elements)
                size = int(obj.shape[0]) * int(obj.shape[1])
                candidates.append((size, name, obj.shape, obj.dtype, obj.chunks))
    f.visititems(visitor)
    if not candidates:
        return None
    candidates.sort(reverse=True, key=lambda x: x[0])
    return candidates[0]  # largest 2D numeric dataset

with h5py.File(FILE_PATH, "r") as f:
    print("\n--- H5 STRUCTURE (first ~1000 nodes) ---")
    print_h5_tree(f)

    best = pick_likely_das_dataset(f)
    if best is None:
        raise ValueError("No 2D numeric dataset found. This H5 may use a different structure.")

    _, DSET_PATH, DSET_SHAPE, DSET_DTYPE, DSET_CHUNKS = best

print("\n✅ Auto-selected dataset:")
print("   DSET_PATH :", DSET_PATH)
print("   shape     :", DSET_SHAPE, "(time, channel) or similar")
print("   dtype     :", DSET_DTYPE)
print("   chunks    :", DSET_CHUNKS)

# Keep these globals for next sections

Drive already mounted at /content/drive
✅ Found H5 file: /content/drive/MyDrive/FiberMind/DAS_Data/10_44_45.h5

--- H5 STRUCTURE (first ~1000 nodes) ---
📁 Acquisition/
  📁 Acquisition/Raw[0]/
    🧊 Acquisition/Raw[0]/RawData  shape=(360152, 1723)  dtype=float32  chunks=(10, 1723)
    🧊 Acquisition/Raw[0]/RawDataSampleCount  shape=(360152,)  dtype=int64  chunks=(1000,)
    🧊 Acquisition/Raw[0]/RawDataTime  shape=(360152,)  dtype=int64  chunks=(1000,)

✅ Auto-selected dataset:
   DSET_PATH : Acquisition/Raw[0]/RawData
   shape     : (360152, 1723) (time, channel) or similar
   dtype     : float32
   chunks    : (10, 1723)


In [ ]:
# ==============================================================================
# SECTION 2 (WINDOWING + OUTPUT DIR UI) — 2048/512 for higher N
# ==============================================================================

import os, json, time, glob
import h5py
import numpy as np
import ipywidgets as widgets
from IPython.display import display
from google.colab import drive

# -----------------------------
# 0) Mount Drive
# -----------------------------
drive.mount("/content/drive")

# -----------------------------
# 1) Paths
# -----------------------------
FILE_PATH = "/content/drive/MyDrive/FiberMind/DAS_Data/10_44_45.h5"
assert os.path.exists(FILE_PATH), f"File not found: {FILE_PATH}"

# Auto-select dataset path (assumes Sintela-style structure)
DSET_PATH = "Acquisition/Raw[0]/RawData"

# -----------------------------
# 2) Windowing params
# -----------------------------
FS_HZ = 100.0
WIN_T = 2048
STRIDE_T = 512

CH_START = 0
CH_COUNT = 1723

# cap for safety (set None for full)
MAX_WINDOWS = None  # e.g., 700 or None

print("\n--- Section 2 config ---")
print("FILE_PATH :", FILE_PATH)
print("DSET_PATH :", DSET_PATH)
print("WIN_T     :", WIN_T)
print("STRIDE_T  :", STRIDE_T)
print("CH_START  :", CH_START)
print("CH_COUNT  :", CH_COUNT)
print("MAX_WINDOWS:", MAX_WINDOWS)
print("FS_HZ     :", FS_HZ)

# -----------------------------
# 3) Validate dataset + compute window count
# -----------------------------
with h5py.File(FILE_PATH, "r") as f:
    assert DSET_PATH in f, f"DSET_PATH not found: {DSET_PATH}"
    dset = f[DSET_PATH]
    T, C = dset.shape
    print("\nDataset shape:", dset.shape, "(time, channel)")
    assert CH_START >= 0 and CH_START < C
    assert CH_COUNT > 0
    ch1 = min(C, CH_START + CH_COUNT)

    max_possible = 1 + max(0, (T - WIN_T) // STRIDE_T)
    if MAX_WINDOWS is None:
        N_WINDOWS = max_possible
    else:
        N_WINDOWS = min(MAX_WINDOWS, max_possible)

print("Max possible windows:", max_possible)
print("Planned windows:", N_WINDOWS)
assert N_WINDOWS > 0, "No windows possible with current WIN_T/STRIDE_T"

# -----------------------------
# 4) Output Dir UI
# -----------------------------
BASE_OUT = "/content/drive/MyDrive/FiberMind/DAS_Outputs"
os.makedirs(BASE_OUT, exist_ok=True)

run_name_default = time.strftime("Run_%Y%m%d_%H%M%S_WIN2048_STR512")
run_text = widgets.Text(value=run_name_default, description="Run name:", layout=widgets.Layout(width="70%"))
btn = widgets.Button(description="Create Output Dir", button_style="success")
out = widgets.Output()

display(run_text, btn, out)

OUTPUT_DIR = None

def on_create(_):
    global OUTPUT_DIR
    with out:
        out.clear_output()
        run_name = run_text.value.strip()
        assert len(run_name) > 0
        OUTPUT_DIR = os.path.join(BASE_OUT, run_name)
        os.makedirs(OUTPUT_DIR, exist_ok=True)
        print("✅ OUTPUT_DIR:", OUTPUT_DIR)

        # Create subdirs (Section 3 will use these)
        os.makedirs(os.path.join(OUTPUT_DIR, "Section3_Preproc_SAFE", "images"), exist_ok=True)
        os.makedirs(os.path.join(OUTPUT_DIR, "Section3_Preproc_SAFE", "arrays"), exist_ok=True)

        # Save manifest now (Section 3 will update it)
        manifest = {
            "run_created": time.strftime("%Y-%m-%d %H:%M:%S"),
            "file_path": FILE_PATH,
            "dataset_path": DSET_PATH,
            "output_dir": OUTPUT_DIR,
            "windowing": {
                "fs_hz": FS_HZ,
                "win_t": WIN_T,
                "stride_t": STRIDE_T,
                "ch_start": CH_START,
                "ch_count": CH_COUNT,
                "n_windows": int(N_WINDOWS),
                "dataset_shape": [int(T), int(C)],
            }
        }
        manifest_path = os.path.join(OUTPUT_DIR, "RUN_MANIFEST.json")
        with open(manifest_path, "w") as f:
            json.dump(manifest, f, indent=2)
        print("✅ Wrote:", manifest_path)

btn.on_click(on_create)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

--- Section 2 config ---
FILE_PATH : /content/drive/MyDrive/FiberMind/DAS_Data/10_44_45.h5
DSET_PATH : Acquisition/Raw[0]/RawData
WIN_T     : 2048
STRIDE_T  : 512
CH_START  : 0
CH_COUNT  : 1723
MAX_WINDOWS: None
FS_HZ     : 100.0

Dataset shape: (360152, 1723) (time, channel)
Max possible windows: 700
Planned windows: 700


Text(value='Run_20260111_162319_WIN2048_STR512', description='Run name:', layout=Layout(width='70%'))

Button(button_style='success', description='Create Output Dir', style=ButtonStyle())

Output()

In [ ]:
# ==============================================================================
# SECTION 3 (SCALE-SAFE): HPF @0.5Hz (sharp) + WPD lvl7 + SAVE EVERY WINDOW (compressed)
# - Saves: arrays (npz) for every window + sparse images for sanity
# - Avoids: RAM blowups
# ==============================================================================

import os, csv, gc, time
import numpy as np
import h5py
import matplotlib.pyplot as plt
import pywt
from scipy import signal
import psutil, json

assert "OUTPUT_DIR" in globals() and OUTPUT_DIR is not None, "Click 'Create Output Dir' in Section 2 first."
assert "FILE_PATH" in globals() and os.path.exists(FILE_PATH)
assert "DSET_PATH" in globals()

SEC3_DIR = os.path.join(OUTPUT_DIR, "Section3_Preproc_SAFE")
IMG_DIR  = os.path.join(SEC3_DIR, "images")
ARR_DIR  = os.path.join(SEC3_DIR, "arrays")
os.makedirs(IMG_DIR, exist_ok=True)
os.makedirs(ARR_DIR, exist_ok=True)
print("✅ Section 3 outputs to:", SEC3_DIR)

# -----------------------------
# RAM monitor
# -----------------------------
def mem_gb():
    return psutil.Process(os.getpid()).memory_info().rss / 1e9

print("Initial RAM (GB):", round(mem_gb(), 2))

# -----------------------------
# Preproc config
# -----------------------------
HP_CUTOFF_HZ = 0.5
HP_ORDER = 8

WAVELET = "db4"
WPD_LEVEL_REQ = 7
WPD_MODE = "symmetric"
THRESHOLD_SCALE = 1.0
THRESH_MODE = "soft"

# Scale knobs (these do NOT change window count)
CH_STRIDE = 4     # subsample channels to reduce WPD burden (change to 2 if you can afford)
CH_MAX = 512      # cap channels
CHANNEL_BLOCK = 32
RAM_STOP_GB = 10.8

# Saving policy (THIS is what fixes your N problem)
SAVE_COMPRESSED_WPD = True
SAVE_WPD_EVERY = 1           # save every window (critical)
SAVE_GRID_EVERY = 20         # images every 20 windows (reduce clutter/runtime)
SHOW_EVERY = 20

CLIP_PCT_LOW = 1.0
CLIP_PCT_HIGH = 99.0

print("\n--- Section 3 config ---")
print("HP:", HP_ORDER, "@", HP_CUTOFF_HZ, "Hz")
print("WPD:", WAVELET, "level", WPD_LEVEL_REQ)
print("CH_STRIDE:", CH_STRIDE, "| CH_MAX:", CH_MAX, "| BLOCK:", CHANNEL_BLOCK)
print("SAVE_WPD_EVERY:", SAVE_WPD_EVERY, "| SAVE_GRID_EVERY:", SAVE_GRID_EVERY)

# -----------------------------
# Filter build
# -----------------------------
sos_hp = signal.butter(HP_ORDER, HP_CUTOFF_HZ, btype="highpass", fs=FS_HZ, output="sos")

def robust_clip(x, p_low=1.0, p_high=99.0):
    lo = np.nanpercentile(x, p_low)
    hi = np.nanpercentile(x, p_high)
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        return x, lo, hi
    return np.clip(x, lo, hi), lo, hi

def wpd_denoise_1d(x):
    max_level = pywt.dwt_max_level(len(x), pywt.Wavelet(WAVELET).dec_len)
    level = int(min(WPD_LEVEL_REQ, max_level))
    if level < 1:
        return x.astype(np.float32), level, 0.0

    wp = pywt.WaveletPacket(data=x, wavelet=WAVELET, mode=WPD_MODE, maxlevel=level)
    nodes = wp.get_level(level, order="freq")

    hf = nodes[-1].data
    sigma = np.median(np.abs(hf - np.median(hf))) / 0.6745 if len(hf) else 0.0
    uthresh = THRESHOLD_SCALE * sigma * np.sqrt(2.0 * np.log(len(x) + 1.0))

    if uthresh > 0:
        for n in nodes:
            n.data = pywt.threshold(n.data, uthresh, mode=THRESH_MODE)

    rec = wp.reconstruct(update=True)[:len(x)]
    return rec.astype(np.float32), level, float(uthresh)

def wpd_block(block):
    T, Cb = block.shape
    out = np.empty_like(block, dtype=np.float32)
    lvl_used, thr_used = None, None
    for c in range(Cb):
        den, lvl, thr = wpd_denoise_1d(block[:, c])
        out[:, c] = den
        if lvl_used is None:
            lvl_used, thr_used = lvl, thr
    return out, lvl_used, thr_used

def save_grid(win_idx, t0, t1, w_raw, w_hp, w_wpd, lvl, thr):
    fig = plt.figure(figsize=(16, 6))
    mats = [
        (w_raw, "RAW (subsampled)"),
        (w_hp,  f"HPF {HP_CUTOFF_HZ}Hz (order {HP_ORDER})"),
        (w_wpd, f"WPD {WAVELET} (lvl {lvl}, thr {thr:.3g})"),
    ]
    for k, (mat, ttl) in enumerate(mats, 1):
        ax = plt.subplot(1, 3, k)
        m2, lo, hi = robust_clip(mat, CLIP_PCT_LOW, CLIP_PCT_HIGH)
        ax.imshow(m2.T, aspect="auto", origin="lower", cmap="viridis")
        ax.set_title(f"{ttl}\nclip[{lo:.2g},{hi:.2g}]")
        ax.set_xlabel("t")
        ax.set_ylabel("ch")
    plt.suptitle(f"win={win_idx} t=[{t0},{t1})", y=1.02)
    plt.tight_layout()
    outp = os.path.join(IMG_DIR, f"win{win_idx:05d}_grid.png")
    fig.savefig(outp, dpi=160, bbox_inches="tight")
    plt.close(fig)
    return outp

# -----------------------------
# Metadata CSV
# -----------------------------
meta_path = os.path.join(SEC3_DIR, "preproc_metadata.csv")
with open(meta_path, "w", newline="") as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow([
        "win_idx","t0","t1",
        "fs_hz","win_t","stride_t",
        "hp_order","hp_cutoff_hz",
        "wavelet","wpd_level_req","wpd_level_used","wpd_thresh",
        "ch_stride","ch_max","channel_block",
        "raw_shape_T","raw_shape_C"
    ])

    # -----------------------------
    # Stream windows from H5
    # -----------------------------
    with h5py.File(FILE_PATH, "r") as f:
        dset = f[DSET_PATH]
        T_total, C_total = dset.shape

        max_possible = 1 + max(0, (T_total - WIN_T) // STRIDE_T)
        N = max_possible if MAX_WINDOWS is None else min(MAX_WINDOWS, max_possible)
        print("Total windows possible:", max_possible, "| Will process:", N)

        t_start = time.time()

        for win_idx in range(N):
            if mem_gb() > RAM_STOP_GB:
                raise RuntimeError(f"Pre-emptive stop: RAM {mem_gb():.2f}GB exceeded {RAM_STOP_GB}GB")

            t0 = win_idx * STRIDE_T
            t1 = t0 + WIN_T
            if t1 > T_total:
                break

            # load raw window
            w_full = np.array(dset[t0:t1, CH_START:CH_START+CH_COUNT], dtype=np.float32)

            # channel subsample + cap
            w_raw = w_full[:, ::CH_STRIDE]
            if w_raw.shape[1] > CH_MAX:
                w_raw = w_raw[:, :CH_MAX]

            # HPF
            w_hp = signal.sosfiltfilt(sos_hp, w_raw, axis=0).astype(np.float32)

            # WPD blocked
            w_wpd = np.empty_like(w_hp, dtype=np.float32)
            C = w_hp.shape[1]
            lvl_used_final, thr_used_final = None, None

            for c0 in range(0, C, CHANNEL_BLOCK):
                c1 = min(C, c0 + CHANNEL_BLOCK)
                den, lvl_used, thr_used = wpd_block(w_hp[:, c0:c1])
                w_wpd[:, c0:c1] = den
                if lvl_used_final is None:
                    lvl_used_final, thr_used_final = lvl_used, thr_used
                del den
                gc.collect()

            # Save compressed WPD array EVERY window
            if SAVE_COMPRESSED_WPD and (win_idx % SAVE_WPD_EVERY == 0):
                out_arr = os.path.join(ARR_DIR, f"win{win_idx:05d}_wpd.npz")
                np.savez_compressed(
                    out_arr,
                    w_wpd=w_wpd,
                    t0=t0, t1=t1,
                    fs_hz=FS_HZ,
                    win_t=WIN_T,
                    stride_t=STRIDE_T,
                    ch_stride=CH_STRIDE,
                    ch_max=CH_MAX
                )

            # Save sparse images
            if (win_idx % SAVE_GRID_EVERY) == 0:
                p = save_grid(win_idx, t0, t1, w_raw, w_hp, w_wpd, lvl_used_final, thr_used_final)
                # print image path occasionally so you can click it in Drive
                print("🖼️ saved:", p)

            writer.writerow([
                win_idx, t0, t1,
                FS_HZ, WIN_T, STRIDE_T,
                HP_ORDER, HP_CUTOFF_HZ,
                WAVELET, WPD_LEVEL_REQ, lvl_used_final, thr_used_final,
                CH_STRIDE, CH_MAX, CHANNEL_BLOCK,
                int(w_raw.shape[0]), int(w_raw.shape[1])
            ])

            if (win_idx % SHOW_EVERY) == 0:
                print(f"win {win_idx}/{N} | RAM={mem_gb():.2f}GB | lvl={lvl_used_final} thr={thr_used_final:.3g}")

            # cleanup
            del w_full, w_raw, w_hp, w_wpd
            plt.close("all")
            gc.collect()

elapsed = time.time() - t_start
print("\n✅ Section 3 complete.")
print("✅ Arrays:", ARR_DIR)
print("✅ Images:", IMG_DIR)
print("✅ Metadata:", meta_path)
print("Runtime (min):", round(elapsed/60, 2))
print("Final RAM (GB):", round(mem_gb(), 2))

# -----------------------------
# Update manifest with Section 3 info
# -----------------------------
manifest_path = os.path.join(OUTPUT_DIR, "RUN_MANIFEST.json")
with open(manifest_path, "r") as f:
    M = json.load(f)

M["section3"] = {
    "section3_dir": SEC3_DIR,
    "arrays_dir": ARR_DIR,
    "images_dir": IMG_DIR,
    "meta_csv": meta_path,
    "hp_cutoff_hz": HP_CUTOFF_HZ,
    "hp_order": HP_ORDER,
    "wavelet": WAVELET,
    "wpd_level_req": WPD_LEVEL_REQ,
    "ch_stride": CH_STRIDE,
    "ch_max": CH_MAX,
    "channel_block": CHANNEL_BLOCK,
    "save_wpd_every": SAVE_WPD_EVERY,
    "save_grid_every": SAVE_GRID_EVERY
}

with open(manifest_path, "w") as f:
    json.dump(M, f, indent=2)

print("✅ Updated manifest:", manifest_path)

✅ Section 3 outputs to: /content/drive/MyDrive/FiberMind/DAS_Outputs/Run_20260111_162319_WIN2048_STR512/Section3_Preproc_SAFE
Initial RAM (GB): 6.19

--- Section 3 config ---
HP: 8 @ 0.5 Hz
WPD: db4 level 7
CH_STRIDE: 4 | CH_MAX: 512 | BLOCK: 32
SAVE_WPD_EVERY: 1 | SAVE_GRID_EVERY: 20
Total windows possible: 700 | Will process: 700
🖼️ saved: /content/drive/MyDrive/FiberMind/DAS_Outputs/Run_20260111_162319_WIN2048_STR512/Section3_Preproc_SAFE/images/win00000_grid.png
win 0/700 | RAM=6.19GB | lvl=7 thr=0.0015
🖼️ saved: /content/drive/MyDrive/FiberMind/DAS_Outputs/Run_20260111_162319_WIN2048_STR512/Section3_Preproc_SAFE/images/win00020_grid.png
win 20/700 | RAM=6.19GB | lvl=7 thr=0.00133
🖼️ saved: /content/drive/MyDrive/FiberMind/DAS_Outputs/Run_20260111_162319_WIN2048_STR512/Section3_Preproc_SAFE/images/win00040_grid.png
win 40/700 | RAM=6.19GB | lvl=7 thr=0.0015
🖼️ saved: /content/drive/MyDrive/FiberMind/DAS_Outputs/Run_20260111_162319_WIN2048_STR512/Section3_Preproc_SAFE/images/win0006

In [ ]:
import os, glob, numpy as np

ARR_DIR = os.path.join(OUTPUT_DIR, "Section3_Preproc_SAFE", "arrays")
IMG_DIR = os.path.join(OUTPUT_DIR, "Section3_Preproc_SAFE", "images")

arrs = sorted(glob.glob(os.path.join(ARR_DIR, "win*_wpd.npz")))
imgs = sorted(glob.glob(os.path.join(IMG_DIR, "*.png")))

print("Arrays saved:", len(arrs))
print("Images saved:", len(imgs))
print("Example array:", arrs[0] if arrs else None)
print("Example image:", imgs[0] if imgs else None)

# Load a random sample and sanity check
if arrs:
    z = np.load(arrs[min(5, len(arrs)-1)])
    w = z["w_wpd"]
    print("Sample w_wpd shape:", w.shape, "dtype:", w.dtype)
    print("min/median/max:", float(np.min(w)), float(np.median(w)), float(np.max(w)))
    print("any NaNs:", np.isnan(w).any())

Arrays saved: 700
Images saved: 35
Example array: /content/drive/MyDrive/FiberMind/DAS_Outputs/Run_20260111_162319_WIN2048_STR512/Section3_Preproc_SAFE/arrays/win00000_wpd.npz
Example image: /content/drive/MyDrive/FiberMind/DAS_Outputs/Run_20260111_162319_WIN2048_STR512/Section3_Preproc_SAFE/images/win00000_grid.png
Sample w_wpd shape: (2048, 431) dtype: float32
min/median/max: -1.2718054056167603 -0.0001366581564070657 1.2518706321716309
any NaNs: False


In [ ]:
import json, os, time

assert "OUTPUT_DIR" in globals() and OUTPUT_DIR is not None

manifest = {
    "run_created": time.strftime("%Y-%m-%d %H:%M:%S"),
    "file_path": FILE_PATH,
    "dataset_path": DSET_PATH,
    "output_dir": OUTPUT_DIR,
    "section3_dir": os.path.join(OUTPUT_DIR, "Section3_Preproc_SAFE"),
    "preproc": {
        "fs_hz": 100.0,
        "win_t": 4096,
        "stride_t": 2048,
        "hp_cutoff_hz": 0.5,
        "hp_order": 8,
        "wavelet": "db4",
        "wpd_level": 7,
        "channel_stride": 4,
        "channel_cap": 512,
        "channel_block": 32,
    }
}

manifest_path = os.path.join(OUTPUT_DIR, "RUN_MANIFEST.json")
with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)

print("✅ Saved run manifest:")
print(manifest_path)

✅ Saved run manifest:
/content/drive/MyDrive/FiberMind/DAS_Outputs/Section2_Windowing_20260111_141812_auto/RUN_MANIFEST.json


In [ ]:
import json, os
from google.colab import drive

drive.mount("/content/drive")

RUN_DIR = "/content/drive/MyDrive/FiberMind/DAS_Outputs/Run_20260111_162319_WIN2048_STR512/"  # paste your folder name here
manifest_path = os.path.join(RUN_DIR, "RUN_MANIFEST.json")

with open(manifest_path, "r") as f:
    M = json.load(f)

print("Loaded manifest from:", manifest_path)
print("H5:", M["file_path"])
print("DSET:", M["dataset_path"])
print("Section 3 dir:", M["section3_dir"])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded manifest from: /content/drive/MyDrive/FiberMind/DAS_Outputs/Section2_Windowing_20260111_141812_auto/RUN_MANIFEST.json
H5: /content/drive/MyDrive/FiberMind/DAS_Data/10_44_45.h5
DSET: Acquisition/Raw[0]/RawData
Section 3 dir: /content/drive/MyDrive/FiberMind/DAS_Outputs/Section2_Windowing_20260111_141812_auto/Section3_Preproc_SAFE


In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [5]:
# ==============================================================================
# SECTION 4 (REGENERATED): MAE FOUNDATION PRETRAIN (PACK -> TRAIN)
# - PACK: Reads Section3 .npz one-by-one from Drive and writes to ONE local memmap file
# - TRAIN: Trains MAE from local memmap (no Drive reads during training loop)
# - RESUME: resumes packing + resumes training from latest checkpoint
# - VISUALS: saves inputs_grid + recon each epoch + loss curve to Drive
# ==============================================================================

import os, glob, json, time, gc
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from google.colab import drive

# -----------------------------
# 0) CONFIG (edit only if your run folder changes)
# -----------------------------
RUN_DIR = "/content/drive/MyDrive/FiberMind/DAS_Outputs/Run_20260111_162319_WIN2048_STR512"
SEC3_ARR_DIR = os.path.join(RUN_DIR, "Section3_Preproc_SAFE", "arrays")

# Local pack directory (runtime local disk)
PACK_DIR = "/content/fibermind_pack"
PACK_MM_PATH = os.path.join(PACK_DIR, "wpd_memmap.dat")
PACK_META_PATH = os.path.join(PACK_DIR, "wpd_pack_meta.json")

# Section 4 output on Drive
SEC4_DIR = os.path.join(RUN_DIR, "Section4_MAE_Foundation")
IMG_DIR  = os.path.join(SEC4_DIR, "images")
CKPT_DIR = os.path.join(SEC4_DIR, "checkpoints")

# Training
EPOCHS_TOTAL = 20          # total epochs to reach (resume will continue up to this)
MASK_RATIO = 0.75
LR = 1e-4
WEIGHT_DECAY = 0.05

# Image sizing (encoder expects 224x224 with patch16)
TARGET_HW = (224, 224)

PRINT_EVERY_STEPS = 20

# -----------------------------
# 1) DEVICE
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("✅ device:", device)

# -----------------------------
# 2) DRIVE MOUNT
# -----------------------------
drive.mount("/content/drive")
assert os.path.isdir(RUN_DIR), f"Missing RUN_DIR: {RUN_DIR}"
assert os.path.isdir(SEC3_ARR_DIR), f"Missing arrays dir: {SEC3_ARR_DIR}"

# -----------------------------
# 3) OUTPUT DIRS
# -----------------------------
os.makedirs(PACK_DIR, exist_ok=True)
os.makedirs(IMG_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)
print("✅ Section 4 outputs to:", SEC4_DIR)

# -----------------------------
# 4) DISCOVER FILES
# -----------------------------
files = sorted(glob.glob(os.path.join(SEC3_ARR_DIR, "win*_wpd.npz")))
assert len(files) > 0, "No win*_wpd.npz files found."
N = len(files)
print("Arrays found:", N)

# Determine T,C from first file (assume consistent)
z0 = np.load(files[0])
w0 = z0["w_wpd"].astype(np.float32)
z0.close()
T, C = w0.shape
print("Window shape (T,C):", (T, C), "dtype=float32")

# -----------------------------
# 5) PACK META (RESUME PACKING)
# -----------------------------
def load_meta():
    if os.path.exists(PACK_META_PATH):
        with open(PACK_META_PATH, "r") as f:
            return json.load(f)
    return None

meta = load_meta()

# If meta missing or mismatch, (re)initialize pack meta and (re)create memmap
if (meta is None) or (meta.get("N") != N) or (meta.get("T") != T) or (meta.get("C") != C) or (not os.path.exists(PACK_MM_PATH)):
    # Fresh pack
    meta = {
        "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
        "run_dir": RUN_DIR,
        "sec3_arr_dir": SEC3_ARR_DIR,
        "N": N, "T": T, "C": C,
        "packed_count": 0,
        "file_list": [os.path.basename(p) for p in files],
        "mm_path": PACK_MM_PATH
    }
    with open(PACK_META_PATH, "w") as f:
        json.dump(meta, f, indent=2)

    # Create new memmap (w+)
    mm = np.memmap(PACK_MM_PATH, dtype="float32", mode="w+", shape=(N, T, C))
    mm.flush()
    del mm
    gc.collect()
    print("🆕 Created fresh memmap pack.")
else:
    print("🔁 Found existing pack meta + memmap. Will resume packing if needed.")

# Attach memmap (r+ to allow writing remaining)
mm = np.memmap(PACK_MM_PATH, dtype="float32", mode="r+", shape=(N, T, C))
packed = int(meta.get("packed_count", 0))
print(f"📦 Pack status: {packed}/{N} packed")

# -----------------------------
# 6) PACK PHASE: NPZ -> MEMMAP (one-by-one, low RAM)
# -----------------------------
if packed < N:
    t0 = time.time()
    for i in range(packed, N):
        z = np.load(files[i])
        w = z["w_wpd"].astype(np.float32)
        z.close()

        mm[i, :, :] = w
        if (i + 1) % 10 == 0 or (i + 1) == N:
            mm.flush()
            meta["packed_count"] = i + 1
            with open(PACK_META_PATH, "w") as f:
                json.dump(meta, f, indent=2)
            print(f"  packed {i+1}/{N}")

        del w
        gc.collect()

    print("✅ Packing complete. Minutes:", round((time.time()-t0)/60, 2))
else:
    print("✅ Packing already complete.")

# Reload meta and validate
with open(PACK_META_PATH, "r") as f:
    meta = json.load(f)
assert meta["packed_count"] == N, "Packing incomplete — rerun this cell to finish packing."

# Important: reopen memmap read-only for training (avoid accidental writes)
del mm
gc.collect()
mm_ro = np.memmap(PACK_MM_PATH, dtype="float32", mode="r", shape=(N, T, C))

# -----------------------------
# 7) NORMALISATION (robust per-window, preserves dynamic range)
# -----------------------------
def robust_unit_scale_np(x, p_low=1.0, p_high=99.0, eps=1e-6):
    lo = np.percentile(x, p_low)
    hi = np.percentile(x, p_high)
    if (not np.isfinite(lo)) or (not np.isfinite(hi)) or hi <= lo:
        m = np.max(np.abs(x)) + eps
        return np.clip(x / m, -1.0, 1.0).astype(np.float32)
    y = np.clip(x, lo, hi)
    y = (y - lo) / (hi - lo + eps)
    return (2.0 * y - 1.0).astype(np.float32)

class MemmapWPDDataset(Dataset):
    def __init__(self, mm, target_hw=(224,224)):
        self.mm = mm
        self.N = mm.shape[0]
        self.target_hw = target_hw

    def __len__(self):
        return self.N

    def __getitem__(self, idx):
        # Copy one window into RAM only (float32)
        w = np.array(self.mm[idx], dtype=np.float32)          # (T,C)
        w = robust_unit_scale_np(w, 1.0, 99.0)

        t = torch.from_numpy(w).unsqueeze(0).unsqueeze(0)     # [1,1,T,C]
        t = F.interpolate(t, size=self.target_hw, mode="bilinear", align_corners=False)
        t = t.squeeze(0)                                      # [1,224,224]
        return t, idx

ds = MemmapWPDDataset(mm_ro, target_hw=TARGET_HW)

batch_size = min(32 if device.type=="cuda" else 8, len(ds))
batch_size = max(1, batch_size)
dl = DataLoader(ds, batch_size=batch_size, shuffle=True, drop_last=False,
                num_workers=0, pin_memory=(device.type=="cuda"))

print("Dataset size:", len(ds))
print("Batch size:", batch_size)
print("Batches/epoch:", len(dl))

# -----------------------------
# 8) VISUAL: INPUT GRID (always titled)
# -----------------------------
def save_inputs_grid(ds, out_path, n=6):
    n = min(n, len(ds))
    fig = plt.figure(figsize=(16,6))
    for i in range(n):
        x,_ = ds[i]
        x = x.squeeze().numpy()
        lo,hi = np.percentile(x,1), np.percentile(x,99)
        x = np.clip(x, lo, hi)
        ax = plt.subplot(2,3,i+1)
        ax.imshow(x, cmap="viridis", origin="lower", aspect="auto")
        ax.set_title(f"Input sample {i}")
        ax.axis("off")
    plt.suptitle("Section 4 — Inputs (display clipped 1–99%)", y=1.02)
    plt.tight_layout()
    plt.savefig(out_path, dpi=180, bbox_inches="tight")
    plt.close()

inputs_grid_path = os.path.join(IMG_DIR, "inputs_grid.png")
if not os.path.exists(inputs_grid_path):
    save_inputs_grid(ds, inputs_grid_path, n=6)
    print("🖼️ saved:", inputs_grid_path)
else:
    print("🖼️ exists:", inputs_grid_path)

# -----------------------------
# 9) MAE MODEL
# -----------------------------
class PatchEmbed(nn.Module):
    def __init__(self, img=224, p=16, in_ch=1, dim=256):
        super().__init__()
        self.p = p
        self.proj = nn.Conv2d(in_ch, dim, kernel_size=p, stride=p)
        self.num_patches = (img//p)*(img//p)
    def forward(self, x):
        x = self.proj(x)                        # [B,D,14,14]
        return x.flatten(2).transpose(1,2)      # [B,196,D]

class Block(nn.Module):
    def __init__(self, dim=256, heads=8, mlp=4.0):
        super().__init__()
        self.n1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.n2 = nn.LayerNorm(dim)
        self.ff = nn.Sequential(
            nn.Linear(dim, int(dim*mlp)),
            nn.GELU(),
            nn.Linear(int(dim*mlp), dim)
        )
    def forward(self, x):
        y,_ = self.attn(self.n1(x), self.n1(x), self.n1(x), need_weights=False)
        x = x + y
        x = x + self.ff(self.n2(x))
        return x

def patchify(imgs, p=16):
    B,C,H,W = imgs.shape
    h = H//p; w = W//p
    x = imgs.reshape(B,C,h,p,w,p).permute(0,2,4,1,3,5).contiguous()
    return x.reshape(B,h*w,C*p*p)              # [B,196,256]

def unpatchify(patches, p=16, img=224):
    B,N,D = patches.shape
    h = img//p; w = img//p
    x = patches.reshape(B,h,w,1,p,p).permute(0,3,1,4,2,5).contiguous()
    return x.reshape(B,1,img,img)

class MAE(nn.Module):
    def __init__(self, img=224, p=16,
                 enc_dim=256, enc_depth=6, enc_heads=8,
                 dec_dim=192, dec_depth=4, dec_heads=6,
                 mask_ratio=0.75):
        super().__init__()
        self.mask_ratio = mask_ratio
        self.p = p

        self.embed = PatchEmbed(img=img, p=p, in_ch=1, dim=enc_dim)
        Np = self.embed.num_patches

        self.pos_e = nn.Parameter(torch.zeros(1, Np, enc_dim))
        self.enc = nn.ModuleList([Block(enc_dim, enc_heads) for _ in range(enc_depth)])
        self.norm_e = nn.LayerNorm(enc_dim)

        self.proj = nn.Linear(enc_dim, dec_dim)
        self.mask_tok = nn.Parameter(torch.zeros(1,1,dec_dim))
        self.pos_d = nn.Parameter(torch.zeros(1, Np, dec_dim))
        self.dec = nn.ModuleList([Block(dec_dim, dec_heads) for _ in range(dec_depth)])
        self.norm_d = nn.LayerNorm(dec_dim)

        self.pred = nn.Linear(dec_dim, p*p)

        nn.init.trunc_normal_(self.pos_e, std=0.02)
        nn.init.trunc_normal_(self.pos_d, std=0.02)
        nn.init.trunc_normal_(self.mask_tok, std=0.02)

    def _mask(self, x):
        B,N,D = x.shape
        keep = max(1, int(N*(1-self.mask_ratio)))
        noise = torch.rand(B, N, device=x.device)
        ids_shuffle = torch.argsort(noise, dim=1)
        ids_restore = torch.argsort(ids_shuffle, dim=1)
        ids_keep = ids_shuffle[:, :keep]
        x_keep = torch.gather(x, 1, ids_keep.unsqueeze(-1).repeat(1,1,D))

        mask = torch.ones(B,N, device=x.device)
        mask[:, :keep] = 0
        mask = torch.gather(mask, 1, ids_restore)
        return x_keep, mask, ids_restore

    def forward(self, imgs):
        x = self.embed(imgs) + self.pos_e
        x_keep, mask, ids_restore = self._mask(x)

        for b in self.enc:
            x_keep = b(x_keep)
        x_keep = self.norm_e(x_keep)

        x_dec = self.proj(x_keep)
        B, nk, D = x_dec.shape
        Np = self.embed.num_patches
        nm = Np - nk
        mask_tokens = self.mask_tok.expand(B, nm, D)
        x_ = torch.cat([x_dec, mask_tokens], dim=1)
        x_ = torch.gather(x_, 1, ids_restore.unsqueeze(-1).repeat(1,1,D))
        x_ = x_ + self.pos_d

        for b in self.dec:
            x_ = b(x_)
        x_ = self.norm_d(x_)
        pred = self.pred(x_)                     # [B,196,256]
        return pred, mask

# -----------------------------
# 10) VISUAL: RECON PER EPOCH (always titled)
# -----------------------------
def save_recon(batch, pred, mask, out_path, title=""):
    x = batch[:1].detach().cpu()                      # [1,1,224,224]
    tgt = patchify(x, p=16)                           # [1,196,256]
    recon = unpatchify(pred[:1].detach().cpu(), p=16, img=224)
    m = mask[:1].detach().cpu().unsqueeze(-1)         # [1,196,1]
    masked = unpatchify(tgt*m, p=16, img=224)

    def disp(a):
        a = a.squeeze().numpy()
        lo,hi = np.percentile(a,1), np.percentile(a,99)
        return np.clip(a, lo, hi)

    fig = plt.figure(figsize=(15,4))
    plt.subplot(1,3,1); plt.imshow(disp(x), cmap="viridis"); plt.title("Input"); plt.axis("off")
    plt.subplot(1,3,2); plt.imshow(disp(masked), cmap="viridis"); plt.title("Masked target"); plt.axis("off")
    plt.subplot(1,3,3); plt.imshow(disp(recon), cmap="viridis"); plt.title("Reconstruction"); plt.axis("off")
    if title:
        plt.suptitle(title, y=1.05)
    plt.tight_layout()
    plt.savefig(out_path, dpi=180, bbox_inches="tight")
    plt.close()

# -----------------------------
# 11) RESUME TRAINING FROM LATEST CHECKPOINT
# -----------------------------
ckpts = sorted(glob.glob(os.path.join(CKPT_DIR, "mae_epoch*.pt")))
model = MAE(mask_ratio=MASK_RATIO).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

losses = []
global_step = 0
start_epoch = 1

if len(ckpts) > 0:
    latest = ckpts[-1]
    ck = torch.load(latest, map_location="cpu")
    model.load_state_dict(ck["model"])
    opt.load_state_dict(ck["opt"])
    losses = ck.get("losses", [])
    start_epoch = int(ck.get("epoch", 0)) + 1
    global_step = len(losses)
    print("🔁 Resuming from:", os.path.basename(latest))
    print("Start epoch:", start_epoch, "| existing steps:", global_step)
else:
    print("🆕 No checkpoints found. Starting fresh.")

if start_epoch > EPOCHS_TOTAL:
    print(f"✅ Already reached epoch >= {EPOCHS_TOTAL}. Nothing to do.")
    raise SystemExit

# AMP (clean API)
use_amp = (device.type == "cuda")
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

# -----------------------------
# 12) TRAIN LOOP
# -----------------------------
model.train()
for epoch in range(start_epoch, EPOCHS_TOTAL + 1):
    saved_recon = False

    for batch, idxs in dl:
        batch = batch.to(device, non_blocking=True)

        with torch.autocast(device_type="cuda", enabled=use_amp):
            pred, mask = model(batch)
            target = patchify(batch, p=16)
            m = mask.unsqueeze(-1)
            loss = ((pred - target)**2 * m).sum() / (m.sum() + 1e-6)

        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        opt.zero_grad(set_to_none=True)

        losses.append(float(loss.detach().cpu()))
        global_step += 1

        if (not saved_recon):
            recon_path = os.path.join(IMG_DIR, f"recon_epoch{epoch:03d}.png")
            title = f"Section 4 — Epoch {epoch}/{EPOCHS_TOTAL} | step {global_step} | loss {losses[-1]:.4g}"
            save_recon(batch, pred, mask, recon_path, title=title)
            print("🖼️ saved:", recon_path)
            saved_recon = True

        if global_step % PRINT_EVERY_STEPS == 0:
            print(f"epoch {epoch}/{EPOCHS_TOTAL} step {global_step} loss={losses[-1]:.6g}")

        del batch, pred, mask, target, m, loss
        gc.collect()

    ckpt_path = os.path.join(CKPT_DIR, f"mae_epoch{epoch:03d}.pt")
    torch.save({"epoch": epoch, "model": model.state_dict(), "opt": opt.state_dict(), "losses": losses}, ckpt_path)
    print("✅ ckpt:", ckpt_path)

# -----------------------------
# 13) LOSS CURVE (NON-BLANK)
# -----------------------------
losses_np = np.array(losses, dtype=np.float64)
plt.figure(figsize=(10,4))
plt.plot(losses_np, linewidth=1)
plt.title("Section 4 — MAE Loss (masked MSE)")
plt.xlabel("step"); plt.ylabel("loss")
v = losses_np[np.isfinite(losses_np)]
if len(v) > 10:
    lo,hi = np.percentile(v, 1), np.percentile(v, 99)
    if hi > lo:
        plt.ylim(lo, hi)
plt.tight_layout()
loss_path = os.path.join(IMG_DIR, "loss_curve.png")
plt.savefig(loss_path, dpi=200, bbox_inches="tight")
plt.close()
print("🖼️ saved:", loss_path)

# -----------------------------
# 14) UPDATE MANIFEST
# -----------------------------
manifest_path = os.path.join(RUN_DIR, "RUN_MANIFEST.json")
try:
    with open(manifest_path, "r") as f:
        M = json.load(f)
except:
    M = {}

M["section4"] = {
    "section4_dir": SEC4_DIR,
    "images_dir": IMG_DIR,
    "checkpoints_dir": CKPT_DIR,
    "epochs_total": EPOCHS_TOTAL,
    "mask_ratio": MASK_RATIO,
    "lr": LR,
    "weight_decay": WEIGHT_DECAY,
    "pack_dir": PACK_DIR,
    "pack_meta": PACK_META_PATH,
    "pack_memmap": PACK_MM_PATH,
    "loss_curve": loss_path,
    "last_ckpt": sorted(glob.glob(os.path.join(CKPT_DIR, "mae_epoch*.pt")))[-1]
}

with open(manifest_path, "w") as f:
    json.dump(M, f, indent=2)

print("✅ Updated manifest:", manifest_path)
print("✅ Section 4 complete.")

✅ device: cuda
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Section 4 outputs to: /content/drive/MyDrive/FiberMind/DAS_Outputs/Run_20260111_162319_WIN2048_STR512/Section4_MAE_Foundation
Arrays found: 700
Window shape (T,C): (2048, 431) dtype=float32
🆕 Created fresh memmap pack.
📦 Pack status: 0/700 packed
  packed 10/700
  packed 20/700
  packed 30/700
  packed 40/700
  packed 50/700
  packed 60/700
  packed 70/700
  packed 80/700
  packed 90/700
  packed 100/700
  packed 110/700
  packed 120/700
  packed 130/700
  packed 140/700
  packed 150/700
  packed 160/700
  packed 170/700
  packed 180/700
  packed 190/700
  packed 200/700
  packed 210/700
  packed 220/700
  packed 230/700
  packed 240/700
  packed 250/700
  packed 260/700
  packed 270/700
  packed 280/700
  packed 290/700
  packed 300/700
  packed 310/700
  packed 320/700
  packed 330/700
  packed 340/700
  packed 350/700
  packed 360/700
  pa

In [6]:
# ==============================================================================
# SECTION 5: FOUNDATION EMBEDDING EXTRACTION + VISUAL VALIDATION
# Inputs:
#   - Section 4 MAE checkpoint
#   - Packed WPD memmap (from Section 4)
# Outputs:
#   - window_embeddings.npy
#   - window_indices.csv
#   - multiple visual diagnostics (saved)
# ==============================================================================

import os, glob, csv, json, gc
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity

# -----------------------------
# 0) CONFIG
# -----------------------------
RUN_DIR = "/content/drive/MyDrive/FiberMind/DAS_Outputs/Run_20260111_162319_WIN2048_STR512"

SEC4_DIR = os.path.join(RUN_DIR, "Section4_MAE_Foundation")
CKPT_DIR = os.path.join(SEC4_DIR, "checkpoints")

PACK_DIR = "/content/fibermind_pack"
PACK_MM_PATH = os.path.join(PACK_DIR, "wpd_memmap.dat")
PACK_META_PATH = os.path.join(PACK_DIR, "wpd_pack_meta.json")

SEC5_DIR = os.path.join(RUN_DIR, "Section5_Embeddings")
IMG_DIR = os.path.join(SEC5_DIR, "images")
os.makedirs(IMG_DIR, exist_ok=True)

TARGET_HW = (224, 224)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("✅ device:", device)

# -----------------------------
# 1) LOAD PACK META
# -----------------------------
with open(PACK_META_PATH, "r") as f:
    meta = json.load(f)

N = meta["N"]
T = meta["T"]
C = meta["C"]

print("Windows:", N, "shape:", (T, C))

mm = np.memmap(PACK_MM_PATH, dtype="float32", mode="r", shape=(N, T, C))

# -----------------------------
# 2) LOAD LATEST MAE CHECKPOINT
# -----------------------------
ckpts = sorted(glob.glob(os.path.join(CKPT_DIR, "mae_epoch*.pt")))
assert len(ckpts) > 0, "No MAE checkpoints found."

ckpt_path = ckpts[-1]
print("🔗 Using checkpoint:", ckpt_path)

ckpt = torch.load(ckpt_path, map_location="cpu")

# -----------------------------
# 3) MAE ENCODER DEFINITION (ENCODER ONLY)
# -----------------------------
class PatchEmbed(nn.Module):
    def __init__(self, img=224, p=16, in_ch=1, dim=256):
        super().__init__()
        self.proj = nn.Conv2d(in_ch, dim, kernel_size=p, stride=p)
        self.num_patches = (img//p)*(img//p)
    def forward(self, x):
        x = self.proj(x)
        return x.flatten(2).transpose(1,2)

class Block(nn.Module):
    def __init__(self, dim=256, heads=8, mlp=4.0):
        super().__init__()
        self.n1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.n2 = nn.LayerNorm(dim)
        self.ff = nn.Sequential(
            nn.Linear(dim, int(dim*mlp)),
            nn.GELU(),
            nn.Linear(int(dim*mlp), dim)
        )
    def forward(self, x):
        y,_ = self.attn(self.n1(x), self.n1(x), self.n1(x), need_weights=False)
        x = x + y
        x = x + self.ff(self.n2(x))
        return x

class MAE_Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = PatchEmbed()
        self.pos = nn.Parameter(torch.zeros(1, self.embed.num_patches, 256))
        self.blocks = nn.ModuleList([Block() for _ in range(6)])
        self.norm = nn.LayerNorm(256)

    def forward(self, x):
        x = self.embed(x) + self.pos
        for b in self.blocks:
            x = b(x)
        x = self.norm(x)
        return x  # [B, N_patches, D]

encoder = MAE_Encoder().to(device)
encoder.load_state_dict({k.replace("embed.","embed.").replace("enc.","blocks.").replace("norm_e","norm"): v
                          for k,v in ckpt["model"].items()
                          if "embed" in k or "enc" in k or "norm_e" in k}, strict=False)

encoder.eval()

# -----------------------------
# 4) ROBUST NORMALISATION (same as training)
# -----------------------------
def robust_unit_scale_np(x, p_low=1.0, p_high=99.0, eps=1e-6):
    lo = np.percentile(x, p_low)
    hi = np.percentile(x, p_high)
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        m = np.max(np.abs(x)) + eps
        return np.clip(x / m, -1.0, 1.0)
    y = np.clip(x, lo, hi)
    y = (y - lo) / (hi - lo + eps)
    return 2*y - 1

# -----------------------------
# 5) EXTRACT EMBEDDINGS
# -----------------------------
embeddings = []
indices = []

BATCH = 16

with torch.no_grad():
    for i in range(0, N, BATCH):
        batch = []
        for j in range(i, min(i+BATCH, N)):
            w = robust_unit_scale_np(mm[j])
            t = torch.from_numpy(w).unsqueeze(0).unsqueeze(0)
            t = F.interpolate(t, size=TARGET_HW, mode="bilinear", align_corners=False)
            batch.append(t)

        batch = torch.cat(batch, dim=0).to(device)
        z = encoder(batch)                     # [B, patches, D]
        z = z.mean(dim=1)                      # global window embedding
        embeddings.append(z.cpu().numpy())
        indices.extend(range(i, min(i+BATCH, N)))

        if i % 100 == 0:
            print("Embedded", i, "/", N)

E = np.concatenate(embeddings, axis=0)
print("Final embeddings:", E.shape)

# -----------------------------
# 6) SAVE OUTPUTS
# -----------------------------
np.save(os.path.join(SEC5_DIR, "window_embeddings.npy"), E)

with open(os.path.join(SEC5_DIR, "window_indices.csv"), "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["index"])
    for i in indices:
        w.writerow([i])

print("✅ Embeddings saved")

# -----------------------------
# 7) VISUAL DIAGNOSTICS
# -----------------------------

# 7.1 Embedding norm histogram
norms = np.linalg.norm(E, axis=1)
plt.figure(figsize=(6,4))
plt.hist(norms, bins=50)
plt.title("Embedding L2 Norm Distribution")
plt.xlabel("||z||"); plt.ylabel("count")
plt.tight_layout()
plt.savefig(os.path.join(IMG_DIR, "embedding_norm_hist.png"), dpi=200)
plt.close()

# 7.2 PCA scatter
pca = PCA(n_components=2)
E2 = pca.fit_transform(E)

plt.figure(figsize=(6,6))
plt.scatter(E2[:,0], E2[:,1], s=8, alpha=0.7)
plt.title("PCA of Window Embeddings")
plt.xlabel("PC1"); plt.ylabel("PC2")
plt.tight_layout()
plt.savefig(os.path.join(IMG_DIR, "pca_scatter.png"), dpi=200)
plt.close()

# 7.3 Cosine similarity heatmap (subset)
S = cosine_similarity(E[:200], E[:200])
plt.figure(figsize=(6,5))
plt.imshow(S, cmap="magma")
plt.title("Cosine Similarity (first 200 windows)")
plt.colorbar()
plt.tight_layout()
plt.savefig(os.path.join(IMG_DIR, "cosine_similarity_heatmap.png"), dpi=200)
plt.close()

# 7.4 Nearest-neighbour retrieval panels
def plot_nn(idx, k=5):
    sims = cosine_similarity(E[idx:idx+1], E)[0]
    nn = np.argsort(-sims)[1:k+1]

    fig = plt.figure(figsize=(15,3))
    for i,j in enumerate([idx]+list(nn)):
        w = mm[j]
        w = robust_unit_scale_np(w)
        lo,hi = np.percentile(w,1), np.percentile(w,99)
        w = np.clip(w, lo, hi)

        ax = plt.subplot(1,k+1,i+1)
        ax.imshow(w.T, aspect="auto", origin="lower", cmap="viridis")
        ax.set_title(f"win {j}")
        ax.axis("off")

    plt.suptitle(f"Nearest Neighbours for window {idx}", y=1.05)
    plt.tight_layout()
    plt.savefig(os.path.join(IMG_DIR, f"nn_window_{idx}.png"), dpi=200)
    plt.close()

for idx in [0, 50, 100, 200, 300]:
    plot_nn(idx)

print("🖼️ Section 5 visuals saved to:", IMG_DIR)
print("✅ Section 5 complete.")

✅ device: cuda
Windows: 700 shape: (2048, 431)
🔗 Using checkpoint: /content/drive/MyDrive/FiberMind/DAS_Outputs/Run_20260111_162319_WIN2048_STR512/Section4_MAE_Foundation/checkpoints/mae_epoch020.pt
Embedded 0 / 700
Embedded 400 / 700
Final embeddings: (700, 256)
✅ Embeddings saved
🖼️ Section 5 visuals saved to: /content/drive/MyDrive/FiberMind/DAS_Outputs/Run_20260111_162319_WIN2048_STR512/Section5_Embeddings/images
✅ Section 5 complete.
